In [18]:
import pandas as pd
import numpy as np

from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_col, summary_params # вывод результатов тестирования

from scipy.stats import f # f-распределение и критические значения
from scipy.stats import t # f-распределение и критические значения
# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# 3.1.1

In [2]:
# импорт данных
df = pd.read_csv('Electricity.csv')

In [15]:
# спецификация модели через формулу
mod = ols(formula='np.log(cost)~1+np.log(q)+I(np.log(q)**2)+np.log(pl)+np.log(pk)+np.log(pf)', data=df)
# подгонка модели с неробастной оценкой ковариационной матрицы
res_ols = mod.fit()
res_ols.params

Intercept           -6.738661
np.log(q)            0.402981
I(np.log(q) ** 2)    0.030440
np.log(pl)           0.146085
np.log(pk)           0.157079
np.log(pf)           0.684705
dtype: float64

In [12]:
summary_params(res_ols, alpha=0.05).round(4)

,Coef.,Std.Err.,t,P>|t|,[0.025,0.975]
Intercept,-7.5118,1.1526,-6.5174,0.0000,-9.7888,-5.2348
np.log(q),0.3704,0.0517,7.1698,0.0000,0.2683,0.4724
I(np.log(q) ** 2),0.0321,0.0035,9.0486,0.0000,0.0251,0.0391
np.log(pk),0.2844,0.0935,3.0410,0.0028,0.0996,0.4691
np.log(pl),0.4476,0.1111,4.0285,0.0001,0.2281,0.6671


In [19]:
# уровень значимости
sign_level = 0.01
# критическое значение t-распределения
t.ppf(q=1-sign_level/2, df=mod.df_resid)

np.float64(2.608560883297947)

In [20]:
summary_params(res_ols, alpha=0.01).round(3)

,Coef.,Std.Err.,t,P>|t|,[0.005,0.995]
Intercept,-6.739,0.706,-9.541,0.000,-8.581,-4.896
np.log(q),0.403,0.032,12.734,0.000,0.320,0.486
I(np.log(q) ** 2),0.030,0.002,14.024,0.000,0.025,0.036
np.log(pl),0.146,0.070,2.073,0.040,-0.038,0.330
np.log(pk),0.157,0.058,2.721,0.007,0.007,0.308
np.log(pf),0.685,0.043,16.043,0.000,0.573,0.796


Значимы: np.log(q)	I(np.log(q) ** 2)	np.log(pk) np.log(pf) т.к. F>Fcr

In [21]:
F_test = res_ols.f_test('np.log(pf)+np.log(pl)+np.log(pk)=1')
print(F_test)

<F test: F=0.014541184876270489, p=0.9041775484872097, df_denom=152, df_num=1>


In [22]:
# Тестовая статистика и её P-значение
F_test.statistic, F_test.pvalue

(0.014541184876270489, np.float64(0.9041775484872097))

In [25]:
# уровень значимости
sign_level = 0.01
# Критическое значение F-распределения
f.isf(q=sign_level, dfn=F_test.df_num, dfd=F_test.df_denom)

np.float64(6.804589881738003)

Не отвергаем гипотезу F < Fcr

In [26]:
# подгонка модели с робастной HC3-оценкой ковариационной матрицы
res_hc = mod.fit(cov_type='HC3')

In [30]:
F_test = res_hc.f_test('np.log(pf)+np.log(pl)+np.log(pk)=1')
# Тестовая статистика и её P-значение
F_test.statistic, F_test.pvalue

(0.013103897835385396, np.float64(0.9090145080300946))

Не отвегаем гипотезу F < Fcr

# 3.1.2

In [31]:
F_test = res_ols.f_test('np.log(pl)=np.log(pk)')
print(F_test)

<F test: F=0.020086275898525834, p=0.8874840609151121, df_denom=152, df_num=1>


In [32]:
# уровень значимости
sign_level = 0.01
# Критическое значение F-распределения
f.isf(q=sign_level, dfn=F_test.df_num, dfd=F_test.df_denom)

np.float64(6.804589881738003)

Не отвергаем, т.к. F < Fcr

In [33]:
# подгонка модели с робастной HC3-оценкой ковариационной матрицы
res_hc = mod.fit(cov_type='HC3')

In [34]:
F_test = res_hc.f_test('np.log(pl)=np.log(pk)')
print(F_test)

<F test: F=0.01710155023323597, p=0.8961278831165537, df_denom=152, df_num=1>


Не отвергаем, т.к. F < Fcr

# 4

## 4.1

### 4.1.1

In [35]:
# импорт данных
df = pd.read_csv('sleep75.csv')

In [37]:
# спецификация исходной модели 
mod = ols(formula='sleep~1+totwrk+age+I(age ** 2) +south+smsa+marr', data = df)
# подгонка исходной модели
res = mod.fit()
# спецификация модели со структурными сдвигами
mod_breaks = ols(formula='sleep~1+totwrk+age+I(age ** 2) +south+smsa+marr+male+totwrk:male+age:male+I(age ** 2):male +south:male+smsa:male+marr:male', data = df)
# подгонка модели со структурными сдвигами
res_breaks = mod_breaks.fit()
# Результаты двух оцениваний в одной таблице
summary_col([res, res_breaks], stars=True, 
            regressor_order=['totwrk', 'age', 'I(age ** 2)','south','smsa', 'marr', 'male', 'totwrk:male', 'age:male','I(age ** 2):male','south:male', 'smsa:male','marr:male'])

,sleep I,sleep II
totwrk,-0.1492***,-0.1400***
,(0.0168),(0.0271)
age,-7.3737,-26.6411
,(11.2528),(17.5840)
I(age ** 2),0.1249,0.3450
,(0.1344),(0.2129)
south,94.4294**,129.1904**
,(41.9101),(61.1601)
smsa,-52.0045,-33.6601
,(33.2032),(49.8399)


In [38]:
F_test = res_breaks.f_test('male=totwrk:male=age:male=I(age ** 2):male=south:male=smsa:male=marr:male=0')
print(F_test)

<F test: F=1.5506147397365047, p=0.1471759143323017, df_denom=692, df_num=7>


In [40]:
# уровень значимости
sign_level = 0.01
# Критическое значение F-распределения
f.isf(q=sign_level, dfn=F_test.df_num, dfd=F_test.df_denom)

np.float64(2.6651528022423494)

F < Fcr, поэтому не отвергаем гипотезу, структурные сдвиги незначимы

In [41]:
# подгонка исходной модели с робастной HC3 ковариационной матрицей
res_hc = mod.fit(cov_type='HC3')
# подгонка модели со структурными сдвигами с робастной HC3 ковариационной матрицей
res_breaks_hc = mod_breaks.fit(cov_type='HC3')
# Результаты двух оцениваний в одной таблице
summary_col([res_hc, res_breaks_hc], stars=True,regressor_order=['totwrk', 'age', 'I(age ** 2)','south','smsa', 'marr', 'male', 'totwrk:male', 'age:male','I(age ** 2):male','south:male', 'smsa:male','marr:male'])

,sleep I,sleep II
totwrk,-0.1492***,-0.1400***
,(0.0192),(0.0307)
age,-7.3737,-26.6411
,(11.5931),(20.2841)
I(age ** 2),0.1249,0.3450
,(0.1353),(0.2389)
south,94.4294**,129.1904*
,(42.2614),(66.0118)
smsa,-52.0045,-33.6601
,(33.6794),(51.0695)


In [42]:
F_test = res_breaks_hc.f_test('male=totwrk:male=age:male=I(age ** 2):male=south:male=smsa:male=marr:male=0')
print(F_test)

<F test: F=1.529911757167221, p=0.15388217893980516, df_denom=692, df_num=7>


F < Fcr значит не отвергаем гипотезу, они незначимые